In [0]:
from pyspark.sql import functions as F

# ------------------------------------------------------------------
# 1) LER A SILVER (dado limpo, 1 linha por óbito)
# ------------------------------------------------------------------
silver = spark.table("silver_mortalidade_sim")
print("Total de linhas na silver:", silver.count())

# ------------------------------------------------------------------
# 2) AGREGAR NA GRANULARIDADE DA FATO
#    (ano, município, sexo, faixa etária, raça/cor, tema, subtema)
# ------------------------------------------------------------------
fato_mortalidade = (
    silver
    .withColumn("codigo_municipio", F.col("CODMUNRES").cast("int"))
    .withColumn("ano", F.col("ANO_REFERENCIA").cast("int"))
    .groupBy("ano", "codigo_municipio", "SEXO_DESC", "FAIXA_ETARIA", "RACACOR_DESC", "TEMA", "SUBTEMA")
    .agg(F.count("*").alias("qtd_obitos"))
)

print("Linhas na fato_mortalidade agregada:", fato_mortalidade.count())

# ------------------------------------------------------------------
# 3) VALIDAÇÃO: a soma de qtd_obitos tem que bater com o total da silver
#    (garante que a agregação não perdeu nem duplicou nenhum óbito)
# ------------------------------------------------------------------
soma_obitos = fato_mortalidade.agg(F.sum("qtd_obitos")).collect()[0][0]
print("Soma de qtd_obitos na fato:", soma_obitos)
print("Bate com o total da silver?", soma_obitos == silver.count())

# ------------------------------------------------------------------
# 4) GRAVAR A TABELA GOLD
# ------------------------------------------------------------------
(fato_mortalidade.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("gold_fato_mortalidade"))

print("Tabela 'gold_fato_mortalidade' gravada com sucesso.")